# Backend API Test Notebook
Use this notebook to smoke-test FastAPI endpoints.

Set `API_BASE` in the next cell to either your Render URL or `http://localhost:3000` for local dev.

> **Note:** The Render free tier spins down after 15 minutes of inactivity. The first request after a cold start can take 30–60 seconds — just wait and retry.


## 0) Start

In [ ]:
import json
from typing import Any
import requests

LOCAL_URL = "http://localhost:3000"
RENDER_URL = "https://london-explorer.onrender.com"  # ← paste your Render URL here
TIMEOUT_SECONDS = 30

# Auto-select: use local if it's up, otherwise fall back to Render
try:
    requests.get(f"{LOCAL_URL}/health", timeout=2)
    API_BASE = LOCAL_URL
    print(f"✓ Local server detected — using {API_BASE}")
except requests.exceptions.ConnectionError:
    API_BASE = RENDER_URL
    print(f"✓ Local server not running — using {API_BASE}")


✓ Local server detected — using http://localhost:3000


In [ ]:


def call_api(path: str, params: dict[str, Any] | None = None) -> Any:
    url = f"{API_BASE}{path}"
    response = requests.get(url, params=params, timeout=TIMEOUT_SECONDS)
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        detail = response.text
        raise requests.HTTPError(f"{exc}\nResponse body: {detail}") from exc
    return response.json()

def preview(payload: Any, max_items: int = 3):
    if isinstance(payload, dict) and "data" in payload and isinstance(payload["data"], list):
        data = payload["data"]
        summary = {k: v for k, v in payload.items() if k != "data"}
        print("Summary:")
        print(json.dumps(summary, indent=2))
        print("\nData preview:")
        print(json.dumps(data[:max_items], indent=2))
        print(f"\nData length: {len(data)}")
        return

    print(json.dumps(payload, indent=2))


## 1) Health Check

In [ ]:
health = call_api("/health")
preview(health)

{
  "status": "ok"
}


## 2) Cuisine Histogram

Verify the cuisine histogram includes the `Unspecified` bucket for places whose `cuisine_type` is SQL `NULL`.

In [ ]:
cuisine_histogram = call_api(
    "/api/cuisine_histogram",
    {"city": "london", "scope": "citywide"},
)

rows = cuisine_histogram["cuisine_histogram"]
unspecified = next((row for row in rows if row["cuisine"] == "Unspecified"), None)

print(f"Cuisine buckets: {len(rows)}")
print(f"Unspecified bucket: {unspecified}")

assert unspecified is not None, "Expected an Unspecified cuisine bucket"
assert unspecified["count"] > 0, "Expected Unspecified cuisine count to be greater than zero"